# 04 - Trading strategy & simulation

Rubric: *Trading simulation (8 pts)* -- the biggest block.

At minimum: two strategies, CAGR/Sharpe/max drawdown, and a benchmark comparison.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from smaz import backtest, utils
from algotrade import ingest, model, strategy, transform

utils.set_plot_defaults()
pd.set_option("display.max_columns", 80)

In [ ]:
df = pd.read_parquet(transform.PROCESSED / "dataset.parquet")
feats = transform.feature_columns(df)
target = transform.TARGET

labelled = df.dropna(subset=[target])
split = model.time_split(labelled, "2021-01-01", "2023-01-01")
clf = model.fit_classifier(split.train, feats, target, kind="xgboost")

test = split.test.copy()
test["proba"] = model.predict_proba(clf, test, feats)

## Strategies

Fees are charged on position changes; signals are lagged one day inside `simulate_signal`.

In [ ]:
test["sig_long"] = strategy.threshold_signal(test["proba"], 0.55)
test["sig_ls"] = strategy.long_short_signal(test["proba"], 0.55, 0.45)
test["sig_top3"] = strategy.top_n_signal(test, n=3)

curves = {
    "long only": strategy.portfolio_returns(test, "sig_long", fee_bps=5),
    "long-short": strategy.portfolio_returns(test, "sig_ls", fee_bps=5),
    "top-3 basket": strategy.portfolio_returns(test, "sig_top3", fee_bps=5),
    "SPY buy&hold": strategy.benchmark_returns(test, ingest.BENCHMARK),
}
strategy.report(curves).round(3)

## Equity curves

In [ ]:
import matplotlib.pyplot as plt

ax = None
for name, r in curves.items():
    ax = backtest.equity_curve(r).plot(ax=ax, label=name)
ax.set_title("Equity curves (test period, net of fees)")
ax.set_ylabel("Growth of $1")
ax.legend()
plt.show()

## Sensitivity & per-ticker breakdown

A point is available for a per-ticker/market breakdown plus improvement ideas.

In [ ]:
# threshold sweep, fee sensitivity, per-ticker contribution
